# 2장 모델 품질 증거 비교

## Goal

모델을 새로 튜닝하지 않고 준비된 canonical evidence에서 baseline, Candidate A, Candidate B의 metric과 release decision을 비교합니다. Candidate A를 보류하고 Candidate B를 승인한 근거를 수치로 설명하는 것이 목표입니다.

## Setup

입력은 `docs/reference/evidence/model/revisions/v2/canonical-benchmark.json`입니다. 이 파일은 sealed test를 한 번 평가한 결과이며 이 Notebook은 evidence를 읽기만 합니다.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "tta-aiqa repository root를 찾지 못했습니다."
{"repository_root": ROOT.name}


{'repository_root': 'tta-aiqa'}

In [2]:
import json
import pandas as pd

EVIDENCE_PATH = ROOT / "docs/reference/evidence/model/revisions/v2/canonical-benchmark.json"
evidence = json.loads(EVIDENCE_PATH.read_text(encoding="utf-8"))
assert evidence["sealed_test"]["status"] == "evaluated_once"
assert evidence["deployment_allowed"] is True
EVIDENCE_PATH.relative_to(ROOT)


PosixPath('docs/reference/evidence/model/revisions/v2/canonical-benchmark.json')

### 1. 동결된 계약 읽기

Canonical evidence는 결과이고, release policy와 model profile은 test를 열기 전에
동결된 판단 입력입니다. 아래 cell은 이 세 artifact를 나란히 읽습니다.

In [3]:
import yaml

POLICY_PATH = ROOT / "configs/qa/revisions/v2.yaml"
PROFILES_PATH = ROOT / "configs/model/revisions/v2/profiles.yaml"
BOOTSTRAP_PATH = ROOT / "docs/reference/evidence/model/revisions/v2/model-bootstrap.json"
policy = yaml.safe_load(POLICY_PATH.read_text(encoding="utf-8"))
profiles = yaml.safe_load(PROFILES_PATH.read_text(encoding="utf-8"))
bootstrap = json.loads(BOOTSTRAP_PATH.read_text(encoding="utf-8"))

pd.DataFrame(
    [
        {
            "profile": profile["name"],
            "model_kind": profile["kind"],
            "threshold": profile["threshold"],
            "model_mlflow_run_id": bootstrap["bundles"][profile["name"]][
                "mlflow_run_id"
            ],
        }
        for profile in profiles["profiles"]
    ]
).set_index("profile")

,model_kind,threshold,model_mlflow_run_id
profile,,,
baseline,logistic_regression,0.50,f4da8c37db114923b5723d220fa61490
candidate-a,random_forest,0.40,f721f26a252e4e4ba465f652e4ca45be
candidate-b,random_forest,0.35,31b50eb011524074960cf0f4e3fd291d


## Steps

### 1. 공통 test metric 비교

In [4]:
decisions = {item["profile"]: item["decision"] for item in evidence["decisions"]}
rows = []
for profile in evidence["profiles"]:
    metrics = profile["metrics"]
    rows.append({
        "profile": profile["profile"],
        "threshold": profile["threshold"],
        "pr_auc": metrics["pr_auc"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "false_negative": metrics["false_negative"],
        "decision": decisions.get(profile["profile"], "REFERENCE"),
    })
comparison = pd.DataFrame(rows).set_index("profile")
comparison.round(4)

,threshold,pr_auc,precision,recall,false_negative,decision
profile,,,,,,
baseline,0.50,0.5244,0.5652,0.2364,42,REFERENCE
candidate-a,0.40,0.5942,0.7727,0.3091,38,HOLD
candidate-b,0.35,0.5743,0.3793,0.8000,11,APPROVE


### 2. Guardrail 수치로 재계산

아래 표는 각 candidate의 metric을 policy의 최소값 또는 baseline 대비 변화와 직접
비교합니다. 이 계산은 판단을 바꾸기 위한 tuning이 아니라 prepared decision을
설명하기 위한 read-only 확인입니다.

In [5]:
baseline_metrics = comparison.loc[policy["baseline_profile"]]
rules = policy["rules"]


def guardrail_rows(profile: str) -> list[dict[str, object]]:
    metrics = comparison.loc[profile]
    return [
        {
            "profile": profile,
            "guardrail": "recall_guardrail",
            "actual": metrics["recall"],
            "required": rules["minimum_recall"] + rules["recall_safety_margin"],
        },
        {
            "profile": profile,
            "guardrail": "recall_uncertainty",
            "actual": next(
                item["bootstrap_recall_lower"]
                for item in evidence["profiles"]
                if item["profile"] == profile
            ),
            "required": rules["minimum_recall_bootstrap_lower"],
        },
        {
            "profile": profile,
            "guardrail": "precision_floor",
            "actual": metrics["precision"],
            "required": rules["minimum_precision"],
        },
        {
            "profile": profile,
            "guardrail": "pr_auc_vs_baseline",
            "actual": metrics["pr_auc"] - baseline_metrics["pr_auc"],
            "required": rules["minimum_pr_auc_delta_vs_baseline"],
        },
        {
            "profile": profile,
            "guardrail": "false_negative_reduction",
            "actual": baseline_metrics["false_negative"] - metrics["false_negative"],
            "required": rules["minimum_false_negative_reduction"],
        },
    ]

policy_comparison = pd.DataFrame(
    guardrail_rows(policy["candidate_a_profile"])
    + guardrail_rows(policy["candidate_b_profile"])
)
policy_comparison["passed"] = policy_comparison["actual"] >= policy_comparison["required"]
policy_comparison.set_index(["profile", "guardrail"]).round(4)

actual  required  passed
profile     guardrail                                          
candidate-a recall_guardrail           0.3091      0.60   False
            recall_uncertainty         0.1864      0.45   False
            precision_floor            0.7727      0.20    True
            pr_auc_vs_baseline         0.0698      0.00    True
            false_negative_reduction   4.0000     15.00   False
candidate-b recall_guardrail           0.8000      0.60    True
            recall_uncertainty         0.6956      0.45    True
            precision_floor            0.3793      0.20    True
            pr_auc_vs_baseline         0.0499      0.00    True
            false_negative_reduction  31.0000     15.00    True

### 2. Release guardrail 확인

PR-AUC 하나만으로 승인하지 않습니다. Recall, uncertainty lower bound, Precision floor와 FN 감소 조건을 함께 확인합니다.

In [6]:
checks = pd.DataFrame({
    item["profile"]: item["checks"] for item in evidence["decisions"]
}).T
checks

,false_negative_reduction,pr_auc_vs_baseline,precision_floor,recall_guardrail,recall_uncertainty
candidate-a,False,True,True,False,False
candidate-b,True,True,True,True,True


## Checks

In [7]:
assert decisions == {"candidate-a": "HOLD", "candidate-b": "APPROVE"}
assert comparison.loc["candidate-b", "recall"] > comparison.loc["baseline", "recall"]
assert comparison.loc["candidate-b", "false_negative"] < comparison.loc["baseline", "false_negative"]
assert not checks.loc["candidate-a"].all()
assert checks.loc["candidate-b"].all()
print("Canonical model evidence checks passed.")

Canonical model evidence checks passed.


## Next Steps

MLflow UI에서 같은 profile의 parameter, validation metric, dataset lineage를 확인합니다. 이 Notebook에서 feature나 threshold를 변경하지 않습니다.